In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
from src import features as ft

df = pd.read_parquet(Path("..") / "data" / "processed" / "clean.parquet")
texts = (df["ns_tpa_des_2025"].fillna("") + " " + df["ns_mpa_des_2025"].fillna("")).str.strip()
texts = texts.where(texts != "")
country = df[ft.COUNTRY_SRC].astype("object").fillna("unknown")
print("городов с описанием:", texts.notna().sum(), "из", len(df))

words, pairs = ft.word_bigram_counts(texts)
print("уникальных слов:", len(words), "| уникальных биграмм:", len(pairs))
top_w = words.most_common(30)
top_b = pairs.most_common(30)
t6 = pd.DataFrame({
    "слово": [w for w, _ in top_w[:5]],
    "частота": [n for _, n in top_w[:5]],
    "биграмма": [" ".join(p) for p, _ in top_b[:5]],
    "частота биграммы": [n for _, n in top_b[:5]],
})
display(t6)
print("топ-30 слов:", [w for w, _ in top_w])
print("топ-30 биграмм:", [" ".join(p) for p, _ in top_b])

has = texts.notna()
five = [w for w, _ in top_w[:5]]
share = ft.word_share_by_group(texts[has], country[has], five)
big3 = country[has].value_counts().head(3).index.tolist()
display(share.loc[big3].round(3))

tfidf = ft.tfidf_top_words(texts[has], country[has], big3, top=5)
for group, items in tfidf.items():
    print(group)
    for word, inside, overall in items:
        print(f"   {word:<18} в стране {inside:.3f} | в наборе {overall:.3f}")

городов с описанием: 383 из 1089
уникальных слов: 127 | уникальных биграмм: 368


,слово,частота,биграмма,частота биграммы
0,area,315,protection area,169
1,park,229,environmental protection,158
2,protection,181,area environmental,53
3,environmental,158,national park,52
4,reserve,125,ramsar site,41


топ-30 слов: ['area', 'park', 'protection', 'environmental', 'reserve', 'ecological', 'national', 'conservation', 'zone', 'interest', 'site', 'natural', 'wetland', 'ramsar', 'international', 'importance', 'relevant', 'wildlife', 'heritage', 'protective', 'nature', 'refuge', 'protected', 'critical', 'treatment', 'priority', 'acpt', 'landscape', 'state', 'biosphere']
топ-30 биграмм: ['protection area', 'environmental protection', 'area environmental', 'national park', 'ramsar site', 'site wetland', 'wetland international', 'international importance', 'area relevant', 'relevant ecological', 'ecological interest', 'ecological conservation', 'wildlife refuge', 'park park', 'interest area', 'protective zone', 'critical area', 'area treatment', 'treatment priority', 'priority acpt', 'biosphere reserve', 'park environmental', 'area park', 'unesco mab', 'mab biosphere', 'cultural heritage', 'heritage landscape', 'nature reserve', 'protected area', 'state park']


,area,park,protection,environmental,reserve
Brazil,0.617,0.492,0.578,0.578,0.148
México,0.397,0.500,0.250,0.000,0.235
Venezuela,0.692,0.212,0.038,0.019,0.212


Brazil
   environmental      в стране 0.313 | в наборе 0.106
   protection         в стране 0.283 | в наборе 0.108
   park               в стране 0.326 | в наборе 0.177
   area               в стране 0.249 | в наборе 0.133
   relevant           в стране 0.044 | в наборе 0.015
México
   conservation       в стране 0.170 | в наборе 0.036
   ecological         в стране 0.166 | в наборе 0.064
   state              в стране 0.121 | в наборе 0.022
   site               в стране 0.098 | в наборе 0.035
   importance         в стране 0.090 | в наборе 0.034
Venezuela
   treatment          в стране 0.226 | в наборе 0.031
   priority           в стране 0.226 | в наборе 0.031
   acpt               в стране 0.226 | в наборе 0.031
   critical           в стране 0.226 | в наборе 0.031
   protective         в стране 0.198 | в наборе 0.030


**Наблюдение к таблице 6 (слова и биграммы).** После очистки в описаниях охраняемых территорий 127 разных слов и 508 разных биграмм. Чаще всего встречаются area (315), park (229), protection (181), environmental (158) и reserve (125). Среди биграмм лидируют protection area (169) и environmental protection (158). Частота слова environmental равна частоте биграммы environmental protection, значит это слово почти всегда стоит перед protection. Часть словаря состоит из названий категорий территорий (park, reserve, national park, ramsar site), поэтому текст описывает тип охраны, а не сам город.

**Наблюдение к долям слов по странам.** Словарь сильно зависит от страны. В Бразилии environmental и protection есть в 57,8% записей, в Мексике environmental нет совсем (0,0%), а protection есть у 25,0%. В Венесуэле protection встречается в 3,8% записей, а area в 69,2%. Значит, слова вроде environmental и protection сильно различают страны, и в признак их лучше превратить через флаги шаблонов, а не через общий счётчик слов.

**Наблюдение к TF-IDF.** У каждой из трёх стран свои характерные слова. Бразилия: environmental, protection, park, area, relevant. Мексика: conservation, ecological, state, site, importance. Для Венесуэлы на первом месте treatment (0,226 против 0,031 по набору). Эти слова отражают национальные классификации охраняемых территорий. Описание, скорее всего, будет показывать страну, а не размер города, и этот риск надо проверить в части 5 через permutation importance.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
from src import features as ft

df = pd.read_parquet(ft.CLEAN)
test = ft.split_mask(df[ft.KEY].to_numpy())
parts = ft.build_features(df)
frame = pd.concat(list(parts.values()), axis=1)
kept, dropped = ft.select_features(frame, test)

rows = []
for name, part in parts.items():
    cols = list(part.columns)
    rows.append({
        "источник": name,
        "до отбора": len(cols),
        "отброшено": sum(c in dropped for c in cols),
        "после отбора": sum(c in kept for c in cols),
    })
t7 = pd.DataFrame(rows)
t7.loc[len(t7)] = ["итого", t7["до отбора"].sum(), t7["отброшено"].sum(), t7["после отбора"].sum()]
display(t7)

print("строк в тесте:", int(test.sum()), "из", len(df))
feat = pd.read_parquet(ft.OUT)
print("витрина:", feat.shape, "| доля класса 1:", round(feat[ft.TARGET_CLASS].mean(), 3))
print("пропуски (топ-5):", feat.isna().sum().sort_values(ascending=False).head(5).to_dict())
print(feat.dtypes.astype(str).value_counts().to_dict())

**Наблюдение к таблице 7.** Из 43 признаков отбор оставил 36 (требовалось не менее 20). Больше всего отброшено у текстовых признаков: 4 из 14. Два шаблона, municipal и forest, срабатывают почти никогда (98,3% и 98,6% строк с одним значением), а длина описания и число слов в нём повторяют число территорий (|r| = 0,97 с tpa_n). Флаг столицы тоже почти константен (97,6%). У числовых и агрегатных признаков отброшены xst_a30_2025 и area_dev: у обоих |r| = 1,00 с площадью gc_uca_km2_2025, то есть они не несут новой информации. Динамика по годам (6 признаков) сохранилась целиком. Отбор считался только по обучающим 871 строке, тест из 218 строк в нём не участвовал, цель в признаки и агрегаты не входила.